In [1]:
!pip install -q transformers torch accelerate sentencepiece


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pickle
import time
import numpy as np
import pandas as pd
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
VECTOR_DIR = "../vector_store"

knowledge_index = faiss.read_index(
    os.path.join(VECTOR_DIR, "knowledge_faiss.index")
)

with open(
    os.path.join(VECTOR_DIR, "knowledge_metadata.pkl"),
    "rb"
) as f:
    knowledge_df = pickle.load(f)

knowledge_df.head()

,source,text
0,billing_policy.txt,Billing Policy Customers are billed according ...
1,business_rules.txt,AI Customer Intelligence Business Rules 1. Cus...
2,cancellation_policy.txt,Cancellation Policy Customers may request canc...
3,retention_policy.txt,Customer Retention Policy Customers identified...
4,service_faq.txt,Service FAQ Customers may subscribe to combina...


In [4]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")
print("Knowledge vectors:", knowledge_index.ntotal)

Embedding model loaded.
Knowledge vectors: 6


In [5]:
def retrieve_knowledge(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = knowledge_index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < len(knowledge_df):
            row = knowledge_df.iloc[idx]

            results.append({
                "score": float(score),
                "source": row["source"],
                "text": row["text"]
            })

    return results

In [6]:
query = "What is the cancellation policy?"

results = retrieve_knowledge(query, top_k=3)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Score:", round(result["score"], 4))
    print("Source:", result["source"])
    print(result["text"])


--- Result 1 ---
Score: 0.6904
Source: cancellation_policy.txt
Cancellation Policy Customers may request cancellation of their service. Customers should review their current contract terms before cancellation. Month-to-month contracts generally provide more flexibility than contracts with longer commitments. Customers considering cancellation may contact support to understand available options and any applicable contractual conditions. The AI assistant must not invent cancellation fees or contract terms. If a specific fee is not present in the knowledge base, the assistant should say that the information is unavailable and recommend checking the customer's contract.

--- Result 2 ---
Score: 0.3451
Source: billing_policy.txt
Billing Policy Customers are billed according to their selected service plan and subscribed services. Monthly charges may include internet service, phone service, streaming services, security services, technical support, and other optional services. Electronic bill

In [7]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

llm.eval()

print("Local LLM loaded successfully.")

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Suthishna kumar\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Xet Storage is ena

Local LLM loaded successfully.


In [8]:
def build_rag_prompt(question, retrieved_results):
    
    context = "\n\n".join(
        [
            f"Source: {r['source']}\n{r['text']}"
            for r in retrieved_results
        ]
    )

    prompt = f"""
You are a customer support assistant.

Answer the user's question using ONLY the provided business knowledge.

Rules:
- Do not invent policies.
- Do not invent customer information.
- If the answer is not available in the knowledge base, say that the information is not available.
- Keep the answer clear and concise.
- Mention the relevant source when useful.

BUSINESS KNOWLEDGE:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    return prompt

In [9]:
def generate_answer(question, top_k=3, max_new_tokens=150):
    
    retrieved_results = retrieve_knowledge(
        question,
        top_k=top_k
    )

    prompt = build_rag_prompt(
        question,
        retrieved_results
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    start_time = time.time()

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    elapsed = time.time() - start_time

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return {
        "question": question,
        "answer": answer.strip(),
        "retrieved_context": retrieved_results,
        "generation_time": elapsed
    }

In [10]:
result = generate_answer(
    "What happens if a customer wants to cancel their service?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nGENERATION TIME:")
print(round(result["generation_time"], 2), "seconds")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION:
What happens if a customer wants to cancel their service?

ANSWER:
If a customer wants to cancel their service, they should first review their current contract terms to ensure it aligns with their needs. They can then contact our support team to discuss available options and any applicable contractual conditions. Please note that we do not offer cancellation fees or specific contract terms beyond what is outlined in the cancellation policy.

GENERATION TIME:
57.6 seconds


In [11]:
test_questions = [
    "What is the billing policy?",
    "What happens if a customer cancels their contract?",
    "What are the available contract options?",
    "How should a high-risk customer be handled?",
    "What should a customer do if they have a technical issue?"
]

rag_results = []

for question in test_questions:
    
    result = generate_answer(question)
    
    rag_results.append({
        "question": question,
        "answer": result["answer"],
        "generation_time": result["generation_time"]
    })
    
    print("=" * 70)
    print("QUESTION:", question)
    print("ANSWER:", result["answer"])
    print("TIME:", round(result["generation_time"], 2), "seconds")

QUESTION: What is the billing policy?
ANSWER: The billing policy states that customers are billed based on their selected service plan and subscribed services. Monthly charges include internet service, phone service, streaming services, security services, technical support, and other optional services. For any billing questions, customers should review their monthly bill and contact customer support for clarification. If there are issues with unexpected charges, reviewing the bill and contacting customer support will help resolve them.
TIME: 64.8 seconds
QUESTION: What happens if a customer cancels their contract?
ANSWER: If a customer cancels their contract, they will need to review their current contract terms before cancellation. The AI assistant recommends contacting support to understand available options and any applicable contractual conditions. It is important to note that the information provided here does not cover all possible scenarios or details, so it is advisable for cus

In [12]:
rag_results_df = pd.DataFrame(rag_results)

os.makedirs("../data/processed", exist_ok=True)

rag_results_df.to_csv(
    "../data/processed/rag_test_results.csv",
    index=False
)

rag_results_df

,question,answer,generation_time
0,What is the billing policy?,The billing policy states that customers are b...,64.799202
1,What happens if a customer cancels their contr...,"If a customer cancels their contract, they wil...",55.333620
2,What are the available contract options?,The cancellation policy states that customers ...,36.941805
3,How should a high-risk customer be handled?,A high-risk customer should receive additional...,52.861047
4,What should a customer do if they have a techn...,A customer should provide details about the te...,21.929410


In [13]:
llm_info = {
    "model": MODEL_NAME,
    "framework": "Hugging Face Transformers",
    "device": "CPU",
    "max_new_tokens": 150,
    "embedding_model": "all-MiniLM-L6-v2"
}

with open(
    "../models/local_llm_info.pkl",
    "wb"
) as f:
    pickle.dump(llm_info, f)

print("LLM configuration saved.")

LLM configuration saved.


In [14]:
def rag_pipeline(question):
    
    result = generate_answer(
        question,
        top_k=3,
        max_new_tokens=150
    )
    
    return {
        "question": question,
        "answer": result["answer"],
        "sources": [
            r["source"]
            for r in result["retrieved_context"]
        ],
        "generation_time": result["generation_time"]
    }

In [15]:
response = rag_pipeline(
    "What is the cancellation policy?"
)

response

{'question': 'What is the cancellation policy?',
 'answer': 'The cancellation policy allows customers to request cancellation of their service. Before canceling, customers should review their current contract terms. For month-to-month contracts, there is more flexibility compared to contracts with longer commitments. Customers considering cancellation should contact support to understand available options and any applicable contractual conditions.',
 'sources': ['cancellation_policy.txt',
  'billing_policy.txt',
  'retention_policy.txt'],
 'generation_time': 46.96207666397095}